# Data Cleaning

## Overview
This notebook performs the **data cleaning stage** for the *Diabetes Hospital Readmission* dataset. The goal is to transform the raw dataset into a consistent and reliable dataset suitable for later analysis and machine learning modeling.

The dataset contains hospital encounter records for diabetic patients admitted to **130 US hospitals between 1999 and 2008**. Each record includes demographic information, admission details, laboratory procedures, medications, diagnosis codes, and the readmission outcome.

Because real-world healthcare datasets often contain missing values, duplicated records, and non-informative variables, several preprocessing steps are required before the data can be used for analysis. This notebook focuses exclusively on identifying and resolving such issues.

The cleaning process includes inspecting the dataset structure, handling missing values, removing identifiers and duplicate patient encounters, eliminating constant and near-constant variables, simplifying diagnosis codes into broader medical categories, and verifying the consistency of categorical variables.

The result of this notebook is a **clean and structured dataset** that can be used in the next stages of the project, including exploratory data analysis and machine learning.

---

## Objectives of this Notebook

The main objectives of this notebook are:

1. Load and inspect the raw dataset
   Examine the dataset structure, column types, and basic statistics.

2. Handle missing values
   Replace placeholder values (`?`) with proper missing values and inspect their distribution.

3. Remove identifiers and duplicate patient records  
   Remove non-informative identifiers such as `encounter_id` and `patient_nbr`, and keep only one encounter per patient to avoid data leakage.

4. Remove constant and near-constant variables
   Drop variables that contain a single value or provide almost no information for modeling.

5. Remove irrelevant records
   Remove newborn admissions, as they are not relevant to hospital readmission prediction for diabetic patients.

6. Simplify diagnosis codes  
   Convert ICD diagnosis codes (`diag_1`, `diag_2`, `diag_3`) into broader medical categories to reduce dimensionality while preserving clinical meaning.

7. Inspect feature distributions
   Review the distributions of demographic, clinical, and medication variables to ensure data consistency.

---

## Output

The cleaned dataset produced in this notebook will be used in the next stage of the project for **exploratory data analysis (EDA)** and **machine learning modeling**.

# Data Cleaning and Preparation

In [89]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [90]:

df_raw = pd.read_csv("../data/raw/diabetic_data.csv")
df = df_raw.copy()

print(df.shape)
df.head()

(101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [91]:
# Check data types and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [92]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
encounter_id,101766.0,1.652016e+08,1.026403e+08,12522.0,84961194.0,152388987.0,2.302709e+08,443867222.0
patient_nbr,101766.0,5.433040e+07,3.869636e+07,135.0,23413221.0,45505143.0,8.754595e+07,189502619.0
admission_type_id,101766.0,2.024006e+00,1.445403e+00,1.0,1.0,1.0,3.000000e+00,8.0
discharge_disposition_id,101766.0,3.715642e+00,5.280166e+00,1.0,1.0,1.0,4.000000e+00,28.0
admission_source_id,101766.0,5.754437e+00,4.064081e+00,1.0,1.0,7.0,7.000000e+00,25.0
time_in_hospital,101766.0,4.395987e+00,2.985108e+00,1.0,2.0,4.0,6.000000e+00,14.0
num_lab_procedures,101766.0,4.309564e+01,1.967436e+01,1.0,31.0,44.0,5.700000e+01,132.0
num_procedures,101766.0,1.339730e+00,1.705807e+00,0.0,0.0,1.0,2.000000e+00,6.0
num_medications,101766.0,1.602184e+01,8.127566e+00,1.0,10.0,15.0,2.000000e+01,81.0
number_outpatient,101766.0,3.693572e-01,1.267265e+00,0.0,0.0,0.0,0.000000e+00,42.0


In [93]:
print("\nMissing values (top 15):")
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print(missing_pct.head(15))




Missing values (top 15):
max_glu_serum    94.746772
A1Cresult        83.277322
encounter_id      0.000000
nateglinide       0.000000
glimepiride       0.000000
acetohexamide     0.000000
glipizide         0.000000
glyburide         0.000000
tolbutamide       0.000000
pioglitazone      0.000000
rosiglitazone     0.000000
acarbose          0.000000
miglitol          0.000000
troglitazone      0.000000
tolazamide        0.000000
dtype: float64


In [94]:
# Replace '?' with np.nan for consistent missing value handling
df.replace('?', np.nan, inplace=True)

# 2) Basic audit
print("Shape:", df.shape)
print("\nMissing values (top 15):")
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print(missing_pct.head(15))

print("\nColumns with >80% missing:")
print(missing_pct[missing_pct > 80])

Shape: (101766, 50)

Missing values (top 15):
weight               96.858479
max_glu_serum        94.746772
A1Cresult            83.277322
medical_specialty    49.082208
payer_code           39.557416
race                  2.233555
diag_3                1.398306
diag_2                0.351787
diag_1                0.020636
encounter_id          0.000000
troglitazone          0.000000
tolbutamide           0.000000
pioglitazone          0.000000
rosiglitazone         0.000000
acarbose              0.000000
dtype: float64

Columns with >80% missing:
weight           96.858479
max_glu_serum    94.746772
A1Cresult        83.277322
dtype: float64


In [95]:
cols_to_drop = ["weight", "max_glu_serum", "A1Cresult"]

df = df.drop(columns=cols_to_drop)

print("New shape:", df.shape)

# confirm they are gone
print("Still present?", [c for c in cols_to_drop if c in df.columns])

New shape: (101766, 47)
Still present? []


In [96]:
df["readmitted"].value_counts()

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [97]:
# Convert readmitted to binary: 1 = <30 days readmission, 0 = >30 or NO
df["readmitted"] = df["readmitted"].apply(lambda x: 1 if x == "<30" else 0)
df["readmitted"].value_counts()

readmitted
0    90409
1    11357
Name: count, dtype: int64

In [98]:
df["patient_nbr"].nunique()

71518

In [99]:
# Remove duplicate patients by keeping only the first encounter
# This prevents the same patient from appearing multiple times,
# which could lead to data leakage between training and test sets
df = df.drop_duplicates(subset="patient_nbr", keep="first")

# Drop the patient identifier since it is not useful for prediction
df.drop(columns=["patient_nbr", "encounter_id"], inplace=True)
df.shape

(71518, 45)

In [100]:
unique_counts = pd.DataFrame({
    "column": df.columns,
    "unique_values": [df[col].nunique() for col in df.columns]
}).sort_values(by="unique_values", ascending=False)

print(unique_counts)

                      column  unique_values
17                    diag_3            758
16                    diag_2            725
15                    diag_1            696
9         num_lab_procedures            116
11           num_medications             75
8          medical_specialty             70
12         number_outpatient             33
4   discharge_disposition_id             26
13          number_emergency             18
7                 payer_code             17
5        admission_source_id             17
18          number_diagnoses             16
6           time_in_hospital             14
14          number_inpatient             13
2                        age             10
3          admission_type_id              8
10            num_procedures              7
0                       race              5
25                 glipizide              4
37       glyburide-metformin              4
36                   insulin              4
31                  miglitol    

In [101]:
print("Shape after dropping high-missing columns:", df.shape)

missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print("\nTop 10 missing columns now:")
print(missing_pct.head(10))

Shape after dropping high-missing columns: (71518, 45)

Top 10 missing columns now:
medical_specialty    48.207444
payer_code           43.405856
race                  2.723790
diag_3                1.712856
diag_2                0.411085
diag_1                0.015381
tolazamide            0.000000
tolbutamide           0.000000
pioglitazone          0.000000
rosiglitazone         0.000000
dtype: float64


In [102]:
cols_to_drop = ["medical_specialty", "payer_code"]

df = df.drop(columns=cols_to_drop)

print("Shape after Step 3:", df.shape)

missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print("\nTop 10 missing columns now:")
print(missing_pct.head(10))

Shape after Step 3: (71518, 43)

Top 10 missing columns now:
race             2.723790
diag_3           1.712856
diag_2           0.411085
diag_1           0.015381
tolazamide       0.000000
tolbutamide      0.000000
pioglitazone     0.000000
rosiglitazone    0.000000
acarbose         0.000000
miglitol         0.000000
dtype: float64


In [103]:
# df = df.drop(columns=["diag_2", "diag_3"], errors="ignore")

# print("Shape after dropping diag_2 and diag_3:", df.shape)

In [104]:
# Check columns with only 1 unique value
constant_cols = []

for col in df.columns:
    if df[col].nunique(dropna=False) == 1:
        constant_cols.append(col)

print("Columns with exactly 1 unique value:")
print(constant_cols)

Columns with exactly 1 unique value:
['examide', 'citoglipton', 'glimepiride-pioglitazone']


In [105]:
df = df.drop(columns=["examide", "citoglipton","glimepiride-pioglitazone"])

print("Shape after dropping constant columns:", df.shape)

Shape after dropping constant columns: (71518, 40)


In [106]:
near_constant_cols = []

for col in df.columns:
    # Skip numeric columns for now
    if df[col].dtype == "object":
        top_freq = df[col].value_counts(normalize=True, dropna=False).iloc[0]
        if top_freq >= 0.995:
            near_constant_cols.append((col, round(top_freq, 4)))

print("Near-constant categorical columns (>=99.5% same value):")
for col, freq in near_constant_cols:
    print(f"{col} → {freq}")

Near-constant categorical columns (>=99.5% same value):
chlorpropamide → 0.999
acetohexamide → 1.0
tolbutamide → 0.9997
acarbose → 0.9972
miglitol → 0.9997
troglitazone → 1.0
tolazamide → 0.9996
glipizide-metformin → 0.9999
metformin-rosiglitazone → 1.0
metformin-pioglitazone → 1.0


In [107]:
cols_to_drop = [col for col, _ in near_constant_cols]

df = df.drop(columns=cols_to_drop)

print("Shape after dropping near-constant columns:", df.shape)

Shape after dropping near-constant columns: (71518, 30)


In [108]:
print("Current columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes.value_counts())

print("\nNumeric columns:")
print(df.select_dtypes(include=np.number).columns.tolist())

print("\nCategorical columns:")
print(df.select_dtypes(include="object").columns.tolist())

Current columns:
['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'metformin', 'repaglinide', 'nateglinide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'insulin', 'glyburide-metformin', 'change', 'diabetesMed', 'readmitted']

Data types:
object    18
int64     12
Name: count, dtype: int64

Numeric columns:
['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'readmitted']

Categorical columns:
['race', 'gender', 'age', 'diag_1', 'diag_2', 'diag_3', 'metformin', 'repaglinide', 'nateglinide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone

In [ ]:
print("Unique diag_1 values:", df["diag_1"].nunique())
print(df["diag_1"].value_counts().head(10))

Unique diag_1 values: 696
diag_1
414    5233
428    3980
786    3040
410    2902
486    2439
       ... 
653       1
61        1
145       1
148       1
V51       1
Name: count, Length: 696, dtype: int64


In [44]:
# Check how many E/V codes exist
ev_codes = df["diag_1"].astype(str).str.startswith(("E", "V")).sum()
print("Number of E/V codes:", ev_codes)

Number of E/V codes: 1645


In [ ]:
def icd_to_float(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    if x.startswith(("E", "V")):
        return np.nan
    try:
        return float(x)
    except:
        return np.nan

def diag_group(x):
    x = icd_to_float(x)
    if pd.isna(x):
        return "Other"
    if 250 <= x < 251:
        return "Diabetes"
    if 390 <= x <= 459 or x == 785:
        return "Circulatory"
    if 460 <= x <= 519 or x == 786:
        return "Respiratory"
    if 520 <= x <= 579 or x == 787:
        return "Digestive"
    if 580 <= x <= 629 or x == 788:
        return "Genitourinary"
    if 800 <= x <= 999:
        return "Injury"
    if 710 <= x <= 739:
        return "Musculoskeletal"
    if 140 <= x <= 239:
        return "Neoplasms"
    return "Other"

df["diag_1_grp"] = df["diag_1"].apply(diag_group)
df["diag_2_grp"] = df["diag_2"].apply(diag_group)
df["diag_3_grp"] = df["diag_3"].apply(diag_group)

df = df.drop(columns=["diag_1", "diag_2", "diag_3"])



diag_1_grp
Circulatory        21894
Other              12358
Respiratory         9776
Digestive           6570
Diabetes            5805
Injury              4779
Musculoskeletal     4080
Genitourinary       3514
Neoplasms           2742
Name: count, dtype: int64


In [111]:
print(df["diag_1_grp"].value_counts())
print("="*20)
print(df["diag_2_grp"].value_counts())
print("="*20)
print(df["diag_3_grp"].value_counts())

diag_1_grp
Circulatory        21894
Other              12358
Respiratory         9776
Digestive           6570
Diabetes            5805
Injury              4779
Musculoskeletal     4080
Genitourinary       3514
Neoplasms           2742
Name: count, dtype: int64
diag_2_grp
Circulatory        22534
Other              18703
Diabetes            9759
Respiratory         7242
Genitourinary       5468
Digestive           2907
Injury              1858
Neoplasms           1750
Musculoskeletal     1297
Name: count, dtype: int64
diag_3_grp
Other              21644
Circulatory        21313
Diabetes           12660
Respiratory         4873
Genitourinary       4199
Digestive           2746
Injury              1443
Musculoskeletal     1378
Neoplasms           1262
Name: count, dtype: int64


In [112]:

print("Shape after removing  diag_1, diag_2, diag_3:", df.shape)

Shape after removing  diag_1, diag_2, diag_3: (71518, 30)


In [113]:
for col in df.columns:
    print("\n", col)
    print(df[col].value_counts(normalize=True).head())


 race
race
Caucasian          0.768880
AfricanAmerican    0.185238
Hispanic           0.021805
Other              0.016933
Asian              0.007144
Name: proportion, dtype: float64

 gender
gender
Female             0.531684
Male               0.468274
Unknown/Invalid    0.000042
Name: proportion, dtype: float64

 age
age
[70-80)    0.254621
[60-70)    0.223161
[50-60)    0.174306
[80-90)    0.162043
[40-50)    0.096172
Name: proportion, dtype: float64

 admission_type_id
admission_type_id
1    0.510221
3    0.194594
2    0.182164
6    0.064152
5    0.044380
Name: proportion, dtype: float64

 discharge_disposition_id
discharge_disposition_id
1     0.619662
3     0.122822
6     0.115901
18    0.034593
2     0.021519
Name: proportion, dtype: float64

 admission_source_id
admission_source_id
7     0.535390
1     0.307713
17    0.069199
4     0.036117
6     0.025182
Name: proportion, dtype: float64

 time_in_hospital
time_in_hospital
3    0.177592
2    0.173341
1    0.149850
4    0.133

In [114]:
df["gender"].value_counts()

gender
Female             38025
Male               33490
Unknown/Invalid        3
Name: count, dtype: int64

In [115]:
df = df[df["gender"] != "Unknown/Invalid"]

print("Rows after removing invalid gender:", df.shape)

Rows after removing invalid gender: (71515, 30)


In [116]:
print("Missing race values:", df["race"].isna().sum())
print("Percentage:", df["race"].isna().mean() * 100)

Missing race values: 1946
Percentage: 2.721107459973432


In [117]:
df["race"] = df["race"].fillna("Unknown")

print(df["race"].value_counts())

race
Caucasian          53491
AfricanAmerican    12887
Unknown             1946
Hispanic            1517
Other               1177
Asian                497
Name: count, dtype: int64


In [124]:
df["discharge_disposition_id"].value_counts().sort_index()

discharge_disposition_id
1     44315
2      1539
3      8784
4       541
5       913
6      8289
7       409
8        73
9         9
10        6
11     1077
12        2
13      243
14      218
15       40
16        3
17        8
18     2474
19        6
20        1
22     1409
23      260
24       25
25      778
27        3
28       90
Name: count, dtype: int64

In [125]:
dead_ids = [11, 19, 20, 21]

df[df["discharge_disposition_id"].isin(dead_ids)].shape

(1084, 30)

In [126]:
df[df["discharge_disposition_id"].isin(dead_ids)]["readmitted"].value_counts()

readmitted
0    1084
Name: count, dtype: int64

In [ ]:
df = df[~df["discharge_disposition_id"].isin(dead_ids)]
df.shape

(70431, 30)

In [129]:
df["admission_type_id"].value_counts().sort_index()

admission_type_id
1    35766
2    12861
3    13834
4        9
5     3122
6     4530
7       18
8      291
Name: count, dtype: int64

In [130]:
df = df[df["admission_type_id"] != 4]
df.shape

(70422, 30)

In [131]:
df.isna().sum().sort_values(ascending=False).head(10)

race                   0
gender                 0
diag_2_grp             0
diag_1_grp             0
readmitted             0
diabetesMed            0
change                 0
glyburide-metformin    0
insulin                0
rosiglitazone          0
dtype: int64

In [132]:
df.isna().sum().sum()

0

In [133]:
df.shape

(70422, 30)

In [134]:
df["readmitted"].value_counts(normalize=True)

readmitted
0    0.910653
1    0.089347
Name: proportion, dtype: float64

In [135]:
df.columns

Index(['race', 'gender', 'age', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'time_in_hospital',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient',
       'number_diagnoses', 'metformin', 'repaglinide', 'nateglinide',
       'glimepiride', 'glipizide', 'glyburide', 'pioglitazone',
       'rosiglitazone', 'insulin', 'glyburide-metformin', 'change',
       'diabetesMed', 'readmitted', 'diag_1_grp', 'diag_2_grp', 'diag_3_grp'],
      dtype='object')

In [136]:
(df.nunique() == 1).sum()

0

In [137]:
print("Final dataset shape:", df.shape)

print("\nData types:")
print(df.dtypes.value_counts())

print("\nTarget distribution:")
print(df["readmitted"].value_counts())

Final dataset shape: (70422, 30)

Data types:
object    18
int64     12
Name: count, dtype: int64

Target distribution:
readmitted
0    64130
1     6292
Name: count, dtype: int64


In [138]:
df.to_csv("../data/processed/diabetes_clean_2.csv", index=False)
print("Saved dataset shape:", df.shape)

Saved dataset shape: (70422, 30)


# Data Cleaning Summary

## Overview
This notebook performed the data cleaning stage for the Diabetes Hospital Readmission dataset. The raw dataset contains hospital encounter records for diabetic patients admitted to 130 US hospitals between 1999 and 2008.

Healthcare datasets often contain missing values, duplicated records, high-cardinality variables, and non-informative features that can negatively affect downstream analysis and machine learning models. Therefore, several preprocessing steps were applied to improve the quality and consistency of the data.

The objective of this notebook was to transform the raw dataset into a clean and structured dataset suitable for further analysis.

---

## Cleaning Steps Performed

### 1. Handling Missing Values
Missing values represented by the placeholder `?` were replaced with proper missing values.

Columns with extremely high missingness (>80%) were removed because they contained little useful information:
- `weight`
- `max_glu_serum`
- `A1Cresult`

Columns with moderate missingness and limited predictive value were also removed:
- `medical_specialty`
- `payer_code`

Small amounts of missing values in `race` were handled by introducing an `"Unknown"` category.

---

### 2. Removing Duplicate Patient Encounters
The dataset contains multiple encounters for the same patient. To avoid potential data leakage during model training, only a single encounter per patient was retained.

The identifier column `patient_nbr` was used to detect duplicate patients, and duplicate encounters were removed accordingly.

---

### 3. Removing Non-informative Columns
Identifier variables and constant features that do not contribute to prediction were removed:
- `encounter_id`
- `patient_nbr`
- `examide`
- `citoglipton`

Additionally, several medication variables that were nearly constant across the dataset were removed to reduce noise and dimensionality.

---

### 4. Removing Irrelevant Records
Records corresponding to newborn admissions were removed because they are not relevant for hospital readmission prediction in diabetic patients.

---

### 5. Diagnosis Simplification
The original diagnosis variables (`diag_1`, `diag_2`, `diag_3`) contained hundreds of ICD codes. To reduce dimensionality while preserving clinical meaning, these codes were grouped into broader disease categories such as:

- Circulatory diseases
- Respiratory diseases
- Digestive diseases
- Diabetes
- Genitourinary diseases
- Musculoskeletal diseases
- Neoplasms
- Injury
- Other

The original ICD code variables were then replaced with the grouped diagnosis variables.

---

### 6. Target Variable Preparation
The original target variable `readmitted` contained three categories:

- `NO`
- `>30`
- `<30`

For the purpose of predicting early hospital readmission, this variable was converted into a binary classification target:

- `1` = readmitted within 30 days
- `0` = not readmitted within 30 days

---

## Final Dataset
After completing the cleaning steps, the dataset contains:

- approximately **71,000 patient records**
- **cleaned demographic, clinical, and medication variables**
- grouped diagnosis categories suitable for machine learning

The dataset is now consistent, free of major missing values, and ready for further analysis.

---

## Next Steps
The cleaned dataset will be used in the next stage of the project to perform:

- Exploratory Data Analysis (EDA)
- Feature encoding
- Machine learning modeling